<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">

# yoctoGPT — Transfer Learning: Pre-train & Fine-tune

**&copy; Dr. Yves J. Hilpisch**<br>AI-Powered by Different LLMs.

This notebook demonstrates **transfer learning** with `yoctoGPT`:
1. Train one BPE tokenizer on a combined corpus (general English + finance texts).
2. **Pre-train** on general English texts.
3. Sample with both a philosophy prompt and a finance prompt.
4. **Fine-tune** on finance texts (warm-start from pre-trained weights).
5. Sample again with the same prompts and compare.

## How to Use This Notebook

- **Goal**: Show how pre-training on a general corpus improves fine-tuning on a domain-specific corpus.
- **Requirements**: Text files in `data/` (general English) and `finance/` (domain-specific).
- **Hardware**: Designed for Google Colab (T4 default). GPU is auto-detected and training parameters are scaled accordingly.
- **Persistence**: Mounts Google Drive for persistent checkpoint storage.
- **GPU Reference**: See `notebooks/colab_gpus.md` for the full GPU comparison table.

### Roadmap

1. **Setup**: Install dependencies, clone repo, mount Google Drive.
2. **Corpus Preparation**: Clean general and finance texts, train a combined BPE tokenizer.
3. **Pre-training**: Train on general English corpus.
4. **Sampling (Pre-trained)**: Generate text with philosophy and finance prompts.
5. **Fine-tuning**: Warm-start from pre-trained checkpoint, train on finance texts.
6. **Sampling (Fine-tuned)**: Same prompts, compare text quality evolution.

In [ ]:
#@title Mount Google Drive for checkpoint storage (repo stays local)
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

CKPT_DIR = Path('/content/drive/MyDrive/yocto/checkpoints/transfer')
CKPT_DIR.mkdir(parents=True, exist_ok=True)
print('Checkpoints dir:', CKPT_DIR)

In [ ]:
#@title Setup: Install Dependencies and Clone Repository
!nvidia-smi || true
!pip -q install tokenizers tqdm textstat pyyaml

import os, pathlib, subprocess

repo_root = pathlib.Path('/content/yoctoGPT')
if repo_root.exists():
    print('Repo exists, pulling latest...')
    subprocess.run(['git', 'pull'], cwd=repo_root, check=False)
else:
    subprocess.run(['git', 'clone', 'https://github.com/yhilpisch/yoctoGPT.git', str(repo_root)], check=False)
os.chdir(repo_root)

if os.path.exists('requirements.txt'):
    !pip -q install -r requirements.txt || true

# Verify both corpora exist
for name, d in [('general', 'data'), ('finance', 'finance')]:
    p = pathlib.Path(d)
    if not p.exists() or not list(p.glob('*.txt')):
        raise FileNotFoundError(f'No .txt files found in {d}/. Add {name} text files before running.')
    txts = sorted(p.glob('*.txt'))
    print(f'Found {len(txts)} {name} text file(s) in {d}/')
    for fp in txts[:5]:
        print(' -', fp.name)
    if len(txts) > 5:
        print(' - ...')

In [ ]:
#@title Detect GPU and load training profile
import subprocess, yaml

_out = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    text=True,
).strip()

if "T4" in _out:
    GPU_KEY = "t4"
elif "L4" in _out:
    GPU_KEY = "l4"
elif "A100" in _out:
    GPU_KEY = "a100"
elif "RTX PRO 6000" in _out:
    GPU_KEY = "g4"
elif "H100" in _out:
    GPU_KEY = "h100"
else:
    GPU_KEY = "t4"

GPU_CFG = yaml.safe_load(open("notebooks/gpu_configs.yml"))[GPU_KEY]
AMP_DTYPE = GPU_CFG["amp_dtype"]
print(f"GPU: {_out} -> profile: {GPU_KEY}, amp_dtype: {AMP_DTYPE}")

### Corpus Preparation

We clean both corpora and train **one BPE tokenizer** on the combined text.
This ensures the tokenizer covers subwords from both general English and finance domains.

Then we encode each corpus separately to create two datasets:
- `data/token_pretrain/` — general English texts for pre-training.
- `data/token_finetune/` — finance texts (formulas dropped) for fine-tuning.

In [ ]:
#@title Clean both corpora and train combined tokenizer
# Clean general texts
!python -m scripts.prepare_book_text \
--in_dir data \
--out_dir data_clean \
--glob '*.txt' \
--lowercase

# Clean finance texts (drop formulas)
!python -m scripts.prepare_book_text \
--in_dir finance \
--out_dir finance_clean \
--glob '*.txt' \
--lowercase \
--drop_formulas

# Combine into one directory for tokenizer training
import shutil, pathlib
combined = pathlib.Path('combined_clean')
if combined.exists():
    shutil.rmtree(combined)
combined.mkdir()
for src_dir in [pathlib.Path('data_clean'), pathlib.Path('finance_clean')]:
    for fp in sorted(src_dir.glob('*.txt')):
        shutil.copy2(fp, combined / fp.name)

# Train tokenizer on combined corpus
!python -m scripts.prepare_tokenizer \
--all_txt_in_dir \
--text_dir combined_clean \
--out_dir data/token_transfer \
--vocab_size 8000 \
--backend bpe \
--random_split \
--split_level chunk \
--split_seed 1337 \
--add_bos_eos

In [ ]:
#@title Encode general and finance texts separately (same tokenizer)
import numpy as np
import pathlib, shutil
from yoctoGPT.tokenizer import load_tokenizer
from yoctoGPT.data import save_ids_bin

tok = load_tokenizer('data/token_transfer/tokenizer.json')
print(f'Loaded tokenizer: vocab_size={tok.vocab_size}')

def encode_dir(text_dir, out_dir, val_ratio=0.1, seed=1337):
    """Encode all .txt files in text_dir, chunk-split, save to out_dir."""
    files = sorted(pathlib.Path(text_dir).glob('*.txt'))
    texts = [p.read_text(encoding='utf-8') for p in files]
    text = '\n\n'.join(texts)
    ids = np.array(tok.encode(text, add_bos=True, add_eos=True), dtype=np.int32)
    rng = np.random.default_rng(seed)
    chunk = 2048
    num_chunks = max(1, (len(ids) + chunk - 1) // chunk)
    chunks = [ids[i*chunk:(i+1)*chunk] for i in range(num_chunks)]
    rng.shuffle(chunks)
    n_val = max(1, min(len(chunks)-1, int(len(chunks) * val_ratio))) if len(chunks) > 1 else 1
    val_ids = np.concatenate(chunks[:n_val])
    train_ids = np.concatenate(chunks[n_val:])
    out = pathlib.Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    save_ids_bin(train_ids, out / 'train.bin')
    save_ids_bin(val_ids, out / 'val.bin')
    # Copy tokenizer so train.py finds it
    shutil.copy2('data/token_transfer/tokenizer.json', out / 'tokenizer.json')
    print(f'{out_dir}: {len(train_ids):,} train, {len(val_ids):,} val tokens ({len(files)} files)')

encode_dir('data_clean', 'data/token_pretrain')
encode_dir('finance_clean', 'data/token_finetune')

In [ ]:
#@title Check corpus sizes and load GPU profile
from pathlib import Path
import numpy as np

for name, d in [('pretrain', 'data/token_pretrain'), ('finetune', 'data/token_finetune')]:
    tp = Path(d) / 'train.bin'
    vp = Path(d) / 'val.bin'
    tt = int(np.fromfile(tp, dtype=np.int32).shape[0])
    vt = int(np.fromfile(vp, dtype=np.int32).shape[0])
    print(f'{name}: {tt:,} train, {vt:,} val tokens')

P = GPU_CFG["token"]
n_layer, n_head = P["n_layer"], P["n_head"]
n_embd = P["n_embd"]
block_size, batch_size = P["block_size"], P["batch_size"]
print(f"\ntoken: n_layer={n_layer}, n_head={n_head}, n_embd={n_embd}, "
      f"block_size={block_size}, batch_size={batch_size}")

### Pre-training

Train on the general English corpus. This gives the model foundational language skills:
grammar, sentence structure, and broad vocabulary.

In [ ]:
#@title Pre-train on general English corpus
PRETRAIN_DIR = CKPT_DIR / 'pretrain'
PRETRAIN_DIR.mkdir(parents=True, exist_ok=True)

!python -m yoctoGPT.train \
--mode token \
--data_dir data/token_pretrain \
--tokenizer_path data/token_pretrain/tokenizer.json \
--ckpt_dir {PRETRAIN_DIR} \
--model_type gpt_fast \
--device cuda \
--n_layer {n_layer} --n_head {n_head} --n_embd {n_embd} \
--block_size {block_size} --batch_size {batch_size} \
--dropout 0.12 --weight_decay 0.08 \
--tie_weights --label_smoothing 0.05 \
--amp --amp_dtype {AMP_DTYPE} \
--auto_microbatch \
--eval_interval 250 --eval_iters 50 \
--cosine_lr --warmup_iters 300 \
--min_lr 1e-5 --lr 1.8e-4 \
--max_iters 2000 \
--ema --ema_decay 0.999

### Sampling (Pre-trained)

We sample with two prompts — one philosophical, one financial — to see how the pre-trained model handles each domain.

In [ ]:
#@title Sample — Philosophy prompt (pre-trained)
PROMPT_PHIL = "In the beginning, philosophy sought to"
output_phil_pre = !python -m yoctoGPT.sampler \
--mode token \
--ckpt {PRETRAIN_DIR}/best.pt \
--tokenizer_path data/token_pretrain/tokenizer.json \
--prompt "{PROMPT_PHIL}" \
--max_new_tokens 120 \
--temperature 0.85 --top_k 40 --top_p 0.90

text_phil_pre = '\n'.join(output_phil_pre)
print(text_phil_pre)

In [ ]:
#@title Sample — Finance prompt (pre-trained)
PROMPT_FIN = "in financial theory, the principle of no arbitrage implies"
output_fin_pre = !python -m yoctoGPT.sampler \
--mode token \
--ckpt {PRETRAIN_DIR}/best.pt \
--tokenizer_path data/token_pretrain/tokenizer.json \
--prompt "{PROMPT_FIN}" \
--max_new_tokens 120 \
--temperature 0.85 --top_k 40 --top_p 0.90

text_fin_pre = '\n'.join(output_fin_pre)
print(text_fin_pre)

### Text Quality Assessment (Pre-trained)

Score both samples against the combined reference corpus.

In [ ]:
#@title Score Generated Text Quality (Pre-trained)
from yoctoGPT.text_metrics import score_text, print_scorecard

reference_corpus = '\n\n'.join(
    p.read_text(encoding='utf-8')
    for p in sorted(pathlib.Path('combined_clean').glob('*.txt'))
)

print('=== Philosophy prompt (pre-trained) ===')
card_phil_pre = score_text(text_phil_pre, reference_text=reference_corpus)
print_scorecard(card_phil_pre)

print('\n=== Finance prompt (pre-trained) ===')
card_fin_pre = score_text(text_fin_pre, reference_text=reference_corpus)
print_scorecard(card_fin_pre)

### Fine-tuning

Warm-start from the pre-trained checkpoint (`--init_from`) and train on finance texts.
The model retains general English skills while adapting to finance vocabulary and style.

Fine-tuning uses a **lower learning rate** (`5e-5` vs `1.8e-4`) to avoid catastrophic forgetting.

In [ ]:
#@title Fine-tune on finance corpus (warm-start from pre-trained)
FINETUNE_DIR = CKPT_DIR / 'finetune'
FINETUNE_DIR.mkdir(parents=True, exist_ok=True)

!python -m yoctoGPT.train \
--mode token \
--data_dir data/token_finetune \
--tokenizer_path data/token_finetune/tokenizer.json \
--ckpt_dir {FINETUNE_DIR} \
--init_from {PRETRAIN_DIR}/best.pt \
--model_type gpt_fast \
--device cuda \
--n_layer {n_layer} --n_head {n_head} --n_embd {n_embd} \
--block_size {block_size} --batch_size {batch_size} \
--dropout 0.12 --weight_decay 0.08 \
--tie_weights --label_smoothing 0.05 \
--amp --amp_dtype {AMP_DTYPE} \
--auto_microbatch \
--eval_interval 200 --eval_iters 60 \
--cosine_lr --warmup_iters 100 \
--min_lr 1e-5 --lr 5e-5 \
--max_iters 1500 \
--ema --ema_decay 0.999

### Sampling (Fine-tuned)

Same two prompts as before. Compare the output with the pre-trained samples to see how fine-tuning changed the model's behavior.

In [ ]:
#@title Sample — Philosophy prompt (fine-tuned)
output_phil_post = !python -m yoctoGPT.sampler \
--mode token \
--ckpt {FINETUNE_DIR}/best.pt \
--tokenizer_path data/token_finetune/tokenizer.json \
--prompt "{PROMPT_PHIL}" \
--max_new_tokens 120 \
--temperature 0.85 --top_k 40 --top_p 0.90

text_phil_post = '\n'.join(output_phil_post)
print(text_phil_post)

In [ ]:
#@title Sample — Finance prompt (fine-tuned)
output_fin_post = !python -m yoctoGPT.sampler \
--mode token \
--ckpt {FINETUNE_DIR}/best.pt \
--tokenizer_path data/token_finetune/tokenizer.json \
--prompt "{PROMPT_FIN}" \
--max_new_tokens 120 \
--temperature 0.85 --top_k 40 --top_p 0.90

text_fin_post = '\n'.join(output_fin_post)
print(text_fin_post)

### Text Quality Assessment (Fine-tuned)

Score both samples and compare with the pre-trained scores above.
The finance prompt should improve significantly; the philosophy prompt should degrade only mildly (or hold steady), demonstrating that the model retained general English skills.

In [ ]:
#@title Score Generated Text Quality (Fine-tuned)
print('=== Philosophy prompt (fine-tuned) ===')
card_phil_post = score_text(text_phil_post, reference_text=reference_corpus)
print_scorecard(card_phil_post)

print('\n=== Finance prompt (fine-tuned) ===')
card_fin_post = score_text(text_fin_post, reference_text=reference_corpus)
print_scorecard(card_fin_post)

print('\n=== Comparison (pre-trained -> fine-tuned) ===')
for label, pre, post in [('Philosophy', card_phil_pre, card_phil_post),
                          ('Finance', card_fin_pre, card_fin_post)]:
    kl_pre = pre.get('Char-KL', float('nan'))
    kl_post = post.get('Char-KL', float('nan'))
    d1_pre = pre.get('Distinct-1', float('nan'))
    d1_post = post.get('Distinct-1', float('nan'))
    print(f'{label:12s}  Char-KL: {kl_pre:.4f} -> {kl_post:.4f}  '
          f'Dist-1: {d1_pre:.4f} -> {d1_post:.4f}')

### Exercises

1. **Fine-tuning Duration**: Reduce `--max_iters` to 800 or increase to 3000. How does the finance prompt quality change? Does the philosophy prompt degrade more with longer fine-tuning?
2. **Learning Rate**: Try `--lr 1e-4` (higher) or `--lr 2e-5` (lower) for fine-tuning. Higher LR adapts faster but may forget more; lower LR preserves general skills but adapts slower.
3. **Tokenizer Size**: Re-run with `--vocab_size 6000` or `10000`. A smaller vocab may generalize better; a larger vocab may capture more domain-specific subwords.
4. **Corpus Mixing**: Instead of sequential pre-train then fine-tune, try training on a mixed corpus (general + finance) in one pass. Compare the results.

<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">